In [ ]:
!pip install requests beautifulsoup4 fitz pypdf pandas tools


Fetching publication details + links to PDF (CORE/arxiv)

In [ ]:
#core --> search/turn into csv
import requests
import csv

API_KEY = "" #core api key from website

def search_core(query, max_results=5):
    url = "https://api.core.ac.uk/v3/search/works"
    headers = {
        "Authorization": f"Bearer {API_KEY}"
    }
    params = {
        "q": query,
        "page": 1,
        "pageSize": max_results
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        print("Failed to retrieve CORE data:", response.status_code, response.text)
        return []

    return response.json().get("results", [])

def save_to_csv(papers, filename="core_results.csv"):
    with open(filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["ID", "Link", "Authors", "Date Published", "Title"])  # header

        for i, paper in enumerate(papers, start=1):
            paper_id = f"core{i}"
            title = paper.get("title", "No Title")
            authors = ", ".join(a.get("name", "Unknown") for a in paper.get("authors", []))
            pdf_url = paper.get("downloadUrl") or paper.get("fullTextUrl") or "No PDF URL found"
            published_date = paper.get("publishedDate") or "No Date"

            writer.writerow([paper_id, pdf_url, authors, published_date, title])

if __name__ == "__main__":
    query = input("Enter your search query for CORE: ")
    papers = search_core(query)
    if papers:
        save_to_csv(papers)
        print(f"\nSaved {len(papers)} results to core_results.csv")


In [ ]:
#arkiv --> searches/convert to csv
import requests
from bs4 import BeautifulSoup
import csv

def search_arxiv_pdf_links(query, max_results=5):
    search_url = f"https://arxiv.org/search/?query={query.replace(' ', '+')}&searchtype=all"
    response = requests.get(search_url)
    if response.status_code != 200:
        print("Failed to retrieve search results")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    results = soup.find_all("li", class_="arxiv-result")

    papers = []
    for i, result in enumerate(results[:max_results], start=1):
        id_tag = result.find("p", class_="list-title is-inline-block")
        title_tag = result.find("p", class_="title is-5 mathjax")
        authors_tag = result.find("p", class_="authors")
        date_tag = result.find("p", class_="is-size-7")

        # ID and link
        if id_tag:
            arxiv_id = id_tag.text.strip().split()[0].replace("arXiv:", "")
            pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
        else:
            arxiv_id = "Unknown"
            pdf_url = "Not found"

        # Title
        title = title_tag.text.strip() if title_tag else "No title"

        # Authors
        if authors_tag:
            authors = [a.text.strip() for a in authors_tag.find_all("a")]
            authors_str = ", ".join(authors)
        else:
            authors_str = "Unknown"

        # Date published
        published_date = date_tag.text.strip().replace("Submitted on", "") if date_tag else "Unknown"

        papers.append([f"ar{i}", pdf_url, authors_str, published_date, title])
        print(f"{pdf_url}")

    return papers

def save_to_csv(papers, filename="arxiv_results.csv"):
    with open(filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["ID", "Link", "Authors", "Date Published", "Title"])
        writer.writerows(papers)

if __name__ == "__main__":
    query = input("Enter search query for arXiv: ")
    papers = search_arxiv_pdf_links(query)
    if papers:
        save_to_csv(papers)
        print(f"\nSaved {len(papers)} results to arxiv_results.csv")


Access links of pdfs and exporting them into csv files (CORE/arxiv)

In [ ]:
#read papers from arxiv and output into arxiv_papers.csv
import pandas as pd
import requests
from io import BytesIO
from pypdf import PdfReader

# Load CSV
df = pd.read_csv("arxiv_results.csv", usecols=["ID", "Link"])
df.columns = df.columns.str.strip()

results = []

for _, row in df.iterrows():
    paper_id = row["ID"]
    pdf_url = row["Link"]

    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()

        # Read the PDF from memory
        reader = PdfReader(BytesIO(response.content))
        full_text = ""
        page_count = len(reader.pages)
        for i, page in enumerate(reader.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
                else:
                    full_text += f"\n[WARNING: Page {i+1} had no extractable text]\n"
            except Exception as page_err:
                full_text += f"\n[ERROR reading page {i+1}: {str(page_err)}]\n"

        full_text = full_text.strip()
        print(f"{paper_id} - Pages: {page_count}, Characters Extracted: {len(full_text)}")
        results.append({"ID": paper_id, "Text": full_text})

    except Exception as e:
        print(f"{paper_id} - ERROR: {str(e)}")
        results.append({"ID": paper_id, "Text": f"ERROR: {str(e)}"})

# Output all results
pd.DataFrame(results).to_csv("arxiv_papers.csv", index=False)


In [ ]:
#read papers from CORE and output into core_papers.csv
import pandas as pd
import requests
from io import BytesIO
from pypdf import PdfReader


df = pd.read_csv("core_results.csv", usecols=["ID", "Link"])
df.columns = df.columns.str.strip()

results = []

for _, row in df.iterrows():
    paper_id = row["ID"]
    pdf_url = row["Link"]

    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()



        reader = PdfReader(BytesIO(response.content))
        full_text = ""
        page_count = len(reader.pages)
        for i, page in enumerate(reader.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
                else:
                    full_text += f"\n[WARNING: Page {i+1} had no extractable text]\n"
            except Exception as page_err:
                full_text += f"\n[ERROR reading page {i+1}: {str(page_err)}]\n"

        full_text = full_text.strip()
        print(f"{paper_id} - Pages: {page_count}, Characters Extracted: {len(full_text)}")
        results.append({"ID": paper_id, "Text": full_text})

    except Exception as e:
        print(f"{paper_id} - ERROR: {str(e)}")
        results.append({"ID": paper_id, "Text": f"ERROR: {str(e)}"})


pd.DataFrame(results).to_csv("core_papers.csv", index=False)


Semantic Scholar (access links and retrieve texts/publication information)

In [ ]:
#semantic papers


import requests
import pandas as pd

def fetch_semantic_papers(query, limit=10):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": limit,
        "fields": "title,abstract,url,authors"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json().get("data", [])

    results = []
    for idx, paper in enumerate(data, 1):
        paper_id = f"sem{idx}"
        title = paper.get("title", "")
        link = paper.get("url", "")
        abstract = paper.get("abstract", "")
        authors = ", ".join([a.get("name", "") for a in paper.get("authors", [])])
        full_text = f"{title}\n\n{abstract}" if abstract else title

        results.append({
            "ID": paper_id,
            "Link": link,
            "Title": title,
            "Authors": authors,
            "Text": full_text.strip()
        })

    return results

if __name__ == "__main__":
    query = input(" Enter a search term for Semantic Scholar: ")
    papers = fetch_semantic_papers(query)
    df = pd.DataFrame(papers)
    df.to_csv("semantic_scholar_papers.csv", index=False)
    print("Saved results to semantic_scholar_papers.csv")


Pubmed 

In [ ]:
#pubmed

import requests
import pandas as pd
from xml.etree import ElementTree as ET

def fetch_pubmed_papers(query, limit=10):
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmax": limit,
        "retmode": "json"
    }
    search_resp = requests.get(search_url, params=search_params)
    search_resp.raise_for_status()
    ids = search_resp.json().get("esearchresult", {}).get("idlist", [])

    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(ids),
        "retmode": "xml"
    }
    fetch_resp = requests.get(fetch_url, params=fetch_params)
    fetch_resp.raise_for_status()
    root = ET.fromstring(fetch_resp.content)

    results = []
    for idx, article in enumerate(root.findall(".//PubmedArticle"), 1):
        paper_id = f"pub{idx}"
        title_elem = article.find(".//ArticleTitle")
        abstract_elems = article.findall(".//AbstractText")
        author_elems = article.findall(".//Author")
        pmid_elem = article.find(".//PMID")

        title = title_elem.text if title_elem is not None else ""
        abstract = " ".join(a.text for a in abstract_elems if a.text)
        authors = []
        for a in author_elems:
            last = a.findtext("LastName")
            first = a.findtext("ForeName")
            if first and last:
                authors.append(f"{first} {last}")
        authors_str = ", ".join(authors)
        pmid = pmid_elem.text if pmid_elem is not None else ""
        link = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"


        full_text = f"{title}\n\n{abstract}" if abstract else title


        results.append({
            "ID": paper_id,
            "Link": link,
            "Title": title,
            "Authors": authors_str,
            "Text": full_text.strip()
        })

    return results

if __name__ == "__main__":
    query = input(" Enter a search term for PubMed: ")
    papers = fetch_pubmed_papers(query)
    df = pd.DataFrame(papers)
    df.to_csv("pubmed_papers.csv", index=False)
    print("Saved results to pubmed_papers.csv")


Clean the texts: 

In [ ]:
import pandas as pd
import re
import os

def clean_research_text(text):
    if not isinstance(text, str):
        return ""

    
    text = re.sub(r'\S+@\S+', '', text)
    
    
    text = re.sub(r'http\S+', '', text)
    
   
    lines = text.split('\n')
    cleaned_lines = []
    prev_line = ""
    for line in lines:
        if line.strip() and line.strip() != prev_line.strip():
            cleaned_lines.append(line)
        prev_line = line
    
    text = '\n'.join(cleaned_lines)
    
    
    lines = text.split('\n')
    filtered_lines = [line for line in lines if not (line.isupper() and len(line.split()) > 3)]
    text = '\n'.join(filtered_lines)
    
    
    boilerplate_phrases = [
        "Recommended Citation", "Follow this and additional works", "All use subject to JSTOR Terms and Conditions",
        "This content downloaded from", "Stable URL", "Linked references are available",
        "You may need to log in to JSTOR", "American Ornithologists' Union",
        "Part of the Biology Commons", "For more information", "JSTOR is a not-for-profit service"
    ]
    for phrase in boilerplate_phrases:
        text = text.replace(phrase, '')
    
    
    text = '\n'.join([line.strip() for line in text.split('\n') if line.strip()])
    
    return text


files_to_clean = [
    ('arxiv_papers.csv', 'Text'),
    ('core_papers.csv', 'Text'),
    ('pubmed_papers.csv', 'Text'),
    ('semantic_scholar_papers.csv', 'Text')
]

for file_path, text_col in files_to_clean:
    if os.path.exists(file_path):
        print(f"Processing '{file_path}' with text column '{text_col}'...")
        df = pd.read_csv(file_path)

        if text_col not in df.columns:
            print(f"Warning: Column '{text_col}' not found in '{file_path}'. Skipping file.")
            continue

       
        df['cleaned_text'] = df[text_col].apply(clean_research_text)

        cleaned_file = file_path.replace('.csv', '_cleaned.csv')
        df.to_csv(cleaned_file, index=False)
        print(f"Saved cleaned file as '{cleaned_file}'.")
    else:
        print(f"File '{file_path}' does not exist. Skipping.")


Move csv files into folders: 

In [ ]:
import os
import shutil

#results.csv files
folder = "results_files"
os.makedirs(folder, exist_ok=True)  
files = ["core_results.csv", "arxiv_results.csv"]
for file in files:
    if os.path.exists(file):
        shutil.move(file, folder)


#papers.csv files   
folder1 = "papers_files"
os.makedirs(folder1, exist_ok=True)
files1 = ["core_papers.csv", "arxiv_papers.csv", "pubmed_papers.csv", "semantic_scholar_papers.csv"]
for file in files1:
    if os.path.exists(file):
        shutil.move(file, folder1)


#cleaned.csv files 
folder2 = "cleaned_files"
os.makedirs(folder2, exist_ok=True)
files2 = ["arxiv_papers_cleaned.csv", "core_papers_cleaned.csv", "pubmed_papers_cleaned.csv", "semantic_scholar_papers_cleaned.csv"]
for file in files2:
    if os.path.exists(file):
        shutil.move(file, folder2)


